## 01 — Feature engineering

The goal of this notebook is to construct **pre-match features** for each tennis match.

A key requirement is to avoid **data leakage**: every historical feature must be computed using only information that would have been available before the match being predicted. The result of the current match is therefore added to the historical data only after its features have been calculated.

Feature described:
1) number of previous matches played by each player
2) each player's previous win rate 
3) performance in their most recent five matches 
4) experience and performance on the current surface 
5) head-to-head performance between the two players
6) ranking and ranking-points difference
7) days since the player's previous match 

Because all historical features depend on past observations, the data must be processed in chronological order. The notebook therefore sorts the matches by date and MatchID before
constructing the features, even if an earlier preprocessing step has already performed this
sorting.

In [7]:
# defaultdict provides dictionaries that automatically provide a default value 
# when a key is encountered for the first time.

# deque stores recent 5 results efficiently

from collections import defaultdict, deque 
from pathlib import Path

import numpy as np
import pandas as pd

In [8]:
# load the preprocessed match data and enforce chronological ordering.
PROCESSED_DATA_DIR = Path("../data/processed")

input_file = PROCESSED_DATA_DIR / "matches_neutral_2015_2025.csv"

matches = pd.read_csv(
    input_file,
    parse_dates=["Date"]
)

# sort by date and MatchID so that all historical features use past information only.
matches = matches.sort_values(
    by=["Date", "MatchID"]
).reset_index(drop=True)

# treat missing surface values as an explicit category
matches["Surface"] = (
    matches["Surface"].fillna("Unknown")
)

print("Dataset shape:", matches.shape)
print("First date:", matches["Date"].min())
print("Last date:", matches["Date"].max())

matches.head()

Dataset shape: (26536, 26)
First date: 2015-01-05 00:00:00
Last date: 2025-11-16 00:00:00


,MatchID,Date,SourceYear,ATP,Location,Tournament,Series,Court,Surface,Round,...,Player2Points,Player1B365Odds,Player2B365Odds,Player1PSOdds,Player2PSOdds,Player1MaxOdds,Player2MaxOdds,Player1AvgOdds,Player2AvgOdds,Player1Won
0,0,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,1195.0,3.50,1.28,3.50,1.34,3.50,1.36,3.30,1.32,0
1,1,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,1730.0,4.50,1.18,4.67,1.23,4.73,1.23,4.31,1.20,1
2,2,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,341.0,1.44,2.62,1.53,2.67,1.53,2.80,1.47,2.62,0
3,3,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,797.0,2.25,1.57,2.37,1.65,2.37,1.67,2.25,1.61,0
4,4,2015-01-05,2015,2,Chennai,Chennai Open,ATP250,Outdoor,Hard,1st Round,...,620.0,1.72,2.00,1.75,2.18,1.80,2.25,1.72,2.07,1


For a player with no previous matches, the ordinary win rate would require division by zero. Moreover, with only one previous match, the ordinary win rate can be either 0 or 1, which may be too extreme given the limited amount of evidence.

We therefore use a smoother win rate, which correponsd to adding one virtual win and one virtual loss. This way a player with no previou smatches receives the neutral value 0.5.

In [9]:
def smoothed_win_rate(wins, matches_played):
    return (wins + 1) / (matches_played + 2)

In [10]:
def recent_win_rate(recent_results ):
    if len(recent_results) == 0:
        return 0.5
    
    return sum(recent_results) / len(recent_results)

# recent_results will contain values such as: [1, 0, 1, 1, 0] 
# which corresponds to the win rate over the last 5 matches
# we return the average

In [11]:
# Historical storage structure: 
# each structure contains only information from already processed dates

player_matches = defaultdict(int) # total number of previous matches
player_wins = defaultdict(int) # total number of previous wins

surface_matches = defaultdict(int) # previous matches by (player, surface)
surface_wins = defaultdict(int) # previous wins by (player, surface)

# keep only the five most recent results for each player.
recent_results = defaultdict(lambda: deque(maxlen=5)) 

# most recent match date for each player
last_match_date = {}

# previous wins in direct encounters, indexed by the ordered pair (winner, loser)
# head_to_head_wins[("Player A", "Player B")] = 3 means player A has previously beaten player B three times.
head_to_head_wins = defaultdict(int)  

# one dictionary per match, containing the original data and the engineered features.
feature_rows = [] 

### Feature and historical updates

For each calendar date, the notebook performs two separate steps:
1. compute every feature using only historical information available before that date
2. after all matches on that date have been processed, add their results to the historical structures

Only matches played before the current date are used to construct historical features, Consequently, matches from earlier dates in 2024 or 2025 can contribute to features for later matches in those years because results would have been known at prediction time.

This is essential to prevent future or same-day match results from influecning the features of another match.

In [12]:
# we first calculate the features using only information available before that match date
# we group by the current date so that we take into consideration only the ealier dates
for current_date, matches_on_date in matches.groupby(
    "Date", 
    sort=True
):
    for _, row in matches_on_date.iterrows():
        player1 = row["Player1"]
        player2 = row["Player2"]
        surface = row["Surface"]
       
        # previous matches before current date
        player1_matches_before = player_matches[player1]
        player2_matches_before = player_matches[player2]
        
        # previous wins before current date 
        player1_wins_before = player_wins[player1]
        player2_wins_before = player_wins[player2]
        
        # smoothed overall win rates
        player1_win_rate = smoothed_win_rate(
            player1_wins_before, 
            player1_matches_before
        )
        
        player2_win_rate = smoothed_win_rate(
            player2_wins_before, 
            player2_matches_before
        )
        
        # win rate over the most recent five matches
        player1_recent_rate = recent_win_rate(
            recent_results[player1]
        )
        
        player2_recent_rate = recent_win_rate(
            recent_results[player2]
        )
        
        # previous performance on the current surface
        player1_surface_matches = surface_matches[
            (player1, surface)
        ]
        
        player2_surface_matches = surface_matches[
            (player2, surface)
        ]
        
        # previous wins on the current surface 
        player1_surface_wins = surface_wins[
            (player1, surface)
        ]
        
        player2_surface_wins = surface_wins[
            (player2, surface)
        ]
        
        # performance rate based on current surface 
        player1_surface_rate = smoothed_win_rate (
            player1_surface_wins, 
            player1_surface_matches
        )
        
        player2_surface_rate = smoothed_win_rate(
            player2_surface_wins,
            player2_surface_matches
        )
        
        # previous head-to-head wins
        player1_h2h_wins = head_to_head_wins[
            (player1, player2)
        ]
        
        player2_h2h_wins = head_to_head_wins[
            (player2, player1)
        ]

        # total number of previous encounters between the two players
        h2h_matches_before = (
            player1_h2h_wins + player2_h2h_wins
        )
        
        # smoothed head-to-head win rates 
        player1_h2h_rate = smoothed_win_rate(
            player1_h2h_wins, 
            h2h_matches_before
        )
        
        player2_h2h_rate = smoothed_win_rate(
            player2_h2h_wins, 
            h2h_matches_before
        )
       
        # days since each player's previous match 
        if player1 in last_match_date: 
            player1_days_since_last_match = (
               current_date - last_match_date[player1]
            ).days
        else:
            player1_days_since_last_match = np.nan
           
        if player2 in last_match_date: 
            player2_days_since_last_match = (
                current_date - last_match_date[player2]
            ).days
        else:
            player2_days_since_last_match = np.nan
            
        # binary indicators for unavailable previous-match dates 
        # 1 indicates that no previous-match date is available, otherwise 0
        player1_missing_days_since_last_match = int(
            pd.isna(player1_days_since_last_match)
        )
        player2_missing_days_since_last_match = int(
            pd.isna(player2_days_since_last_match)
        )
        
        # difference in days since the players' last match
        if(
            player1_missing_days_since_last_match == 0
            and player2_missing_days_since_last_match == 0
        ):
            days_since_last_match_difference = (
                player1_days_since_last_match - player2_days_since_last_match
            )
        else:
            days_since_last_match_difference = np.nan

        # start from the original match row 
        feature_row = row.to_dict()
        
        # add the newly engineered pre-match features 
        feature_row.update(
            {
                "Player1MatchesBefore": player1_matches_before, 
                
                "Player2MatchesBefore": player2_matches_before,
                
                "ExperienceDifference": player1_matches_before - player2_matches_before, 
                
                "Player1WinRateBefore": player1_win_rate, 
                
                "Player2WinRateBefore": player2_win_rate,
                
                "WinRateDifference": player1_win_rate - player2_win_rate,
                
                "Player1Recent5WinRate": player1_recent_rate, 
                
                "Player2Recent5WinRate": player2_recent_rate, 
                
                "Recent5WinRateDifference": player1_recent_rate - player2_recent_rate, 
                
                "Player1SurfaceMatchesBefore": player1_surface_matches,
                
                "Player2SurfaceMatchesBefore": player2_surface_matches, 
                
                "SurfaceMatchesDifference": player1_surface_matches - player2_surface_matches,
                
                "Player1SurfaceWinRateBefore": player1_surface_rate, 
                
                "Player2SurfaceWinRateBefore": player2_surface_rate, 
                
                "SurfaceWinRateDifference": player1_surface_rate - player2_surface_rate, 
                
                "H2HMatchesBefore": h2h_matches_before, 
                
                "Player1H2HWinRateBefore": player1_h2h_rate, 
                
                "Player2H2HWinRateBefore": player2_h2h_rate,
                
                "DifferenceH2HWinRateBefore": player1_h2h_rate - player2_h2h_rate, 
                
                "Player1DaysSinceLastMatch": player1_days_since_last_match,
                
                "Player2DaysSinceLastMatch": player2_days_since_last_match, 
                
                "DaysSinceLastMatchDifference": days_since_last_match_difference,
                
                "MissingPlayer1DaysSinceLastMatch": player1_missing_days_since_last_match,
                
                "MissingPlayer2DaysSinceLastMatch": player2_missing_days_since_last_match,
                
                "RankDifference": row["Player2Rank"] - row["Player1Rank"], 
                
                "PointsDifference": row["Player1Points"] - row["Player2Points"]
            }
        )
        feature_rows.append(feature_row)

    # we've calculated the features
    # now we update the historical statistics with the matches that happened on that date
    for _, row in matches_on_date.iterrows(): 
        
        player1= row["Player1"]
        player2 = row["Player2"]
        surface = row["Surface"]

        # first determine winner and loser
        if row["Player1Won"] == 1: # if player 1 won
            winner = player1
            loser = player2
            
            player1_result = 1
            player2_result = 0
        else:                      # if player 2 won
            winner = player2
            loser = player1
            
            player1_result = 0
            player2_result = 1

        # then we update the historical information    
        # update total match count 
        player_matches[player1] +=1
        player_matches[player2] +=1
        
        # update total win counts
        player_wins[winner] +=1
        
        # update surface-specific match counts 
        surface_matches[(player1, surface)] +=1
        surface_matches[(player2, surface)] +=1
        
        # update surface-specific win counts
        surface_wins[(winner, surface)] +=1
        
        # update each player's recent-results history
        recent_results[player1].append(player1_result)
        recent_results[player2].append(player2_result)
        
        # update each player's most recent match date 
        last_match_date[player1] = current_date 
        last_match_date[player2] = current_date
        
        # update head-to-head history
        head_to_head_wins[(winner, loser)] +=1
        
        

Rather than simply giving the single statistics for each player regarding features such as the last match played or the surface win rate. We also add the difference between the values of each player (e.g. WinRateDifference = Player1WinRate − Player2WinRate).

This way we can directly represent the advantage of Player 1 relative to Player 2. 
- A positive value means Player 1 has the advantage.
- A negative value means Player 2 has the advantage.

In [13]:
# turn list with a dictionary for each match into dataframe
featured_matches = pd.DataFrame(feature_rows)

print("Original shape:", matches.shape)
print("Featured shape:", featured_matches.shape)

Original shape: (26536, 26)
Featured shape: (26536, 52)


### Inspecting the new features 

For matches on the earliest available date, the players have no previous observations in the dataset. We therefore expect the historical features to take their neutral or initial values

ex: MatchesBefore = 0, 
    WinRateBefore = 0.5
    Recent5WinRate = 0.5
    SurfaceWinRate = 0.5
    H2HWinRate = 0.5, etc.

the days-since-last-match features are NaN for players with no previous match in the available dataset; the corresponding binary missing indicators are set to 1.

In [14]:
# list the engineered features and inspect their values for the first few matches
feature_columns = [
    "Player1MatchesBefore", 
    "Player2MatchesBefore",
    "ExperienceDifference",
    
    "Player1WinRateBefore",
    "Player2WinRateBefore",
    "WinRateDifference",
    
    "Player1Recent5WinRate",
    "Player2Recent5WinRate",
    "Recent5WinRateDifference",
    
    "Player1SurfaceMatchesBefore",
    "Player2SurfaceMatchesBefore",
    "SurfaceMatchesDifference",
    
    "Player1SurfaceWinRateBefore",
    "Player2SurfaceWinRateBefore",
    "SurfaceWinRateDifference",
    
    "H2HMatchesBefore",
    
    "Player1H2HWinRateBefore",
    "Player2H2HWinRateBefore",
    "DifferenceH2HWinRateBefore",
    
    "Player1DaysSinceLastMatch",
    "Player2DaysSinceLastMatch",
    "DaysSinceLastMatchDifference",
    
    "MissingPlayer1DaysSinceLastMatch",
    "MissingPlayer2DaysSinceLastMatch",
    
    "RankDifference",
    "PointsDifference"
]

featured_matches[
    [
        "Date",
        "Player1",
        "Player2",
        "Surface",
        "Player1Won"
    ] + feature_columns
].head(10)

,Date,Player1,Player2,Surface,Player1Won,Player1MatchesBefore,Player2MatchesBefore,ExperienceDifference,Player1WinRateBefore,Player2WinRateBefore,...,Player1H2HWinRateBefore,Player2H2HWinRateBefore,DifferenceH2HWinRateBefore,Player1DaysSinceLastMatch,Player2DaysSinceLastMatch,DaysSinceLastMatchDifference,MissingPlayer1DaysSinceLastMatch,MissingPlayer2DaysSinceLastMatch,RankDifference,PointsDifference
0,2015-01-05,Golubev A.,Chardy J.,Hard,0,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,-41.0,-504.0
1,2015-01-05,Duckworth J.,Simon G.,Hard,1,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,-104.0,-1300.0
2,2015-01-05,Benneteau J.,Kokkinakis T.,Hard,0,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,124.0,1024.0
3,2015-01-05,Querrey S.,Tomic B.,Hard,0,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,18.0,293.0
4,2015-01-05,Coric B.,Haase R.,Hard,1,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,-15.0,-63.0
5,2015-01-05,Roger-Vasselin E.,Muller G.,Hard,0,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,-70.0,-385.0
6,2015-01-05,Becker B.,Bolelli S.,Hard,0,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,12.0,163.0
7,2015-01-05,Lorenzi P.,Brown D.,Hard,0,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,35.0,210.0
8,2015-01-05,Dodig I.,Safwat M.,Hard,1,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,200.0,407.0
9,2015-01-05,Gasquet R.,Andujar P.,Hard,1,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,15.0,400.0


### Definition of difference features

The differencce features are oriented from Player 1's perspective:
- RANK DIFFERENCE = player2Rank - player1Rank
    Tennis rankings use smaller numbers for better positions. Therefore:
    1) if RankDifference is positive --> Player1 has better ranking 
    2) else RankDifference is negative --> Player2 has better ranking
- POINTS DIFFERENCE  = player1Points - player2Points
    A positive value means player 1 has more ranking points. 

The same Player1 - Player2 convention is used for the performance_based difference features, such as WinRateDifference and SurfaceWinRateDifference.

### Validation checks 

Before saving the engineered dataset, we verify several basic consistency conditions:
1) Player1Won must always have value 0 or 1
2) match ID remains unique
3) every Rate calculation must be within the interval [0, 1]
4) missing-value indicators for days since the last match are binary
5) number of rows is unchanged after feature engineering

These help detect errors in the feature-construction process before we move onto the modelling step.

In [15]:
# check row preservation, target validity, uniqueness, and feature ranges.
assert len(featured_matches) == len(matches)

assert featured_matches["MatchID"].is_unique

assert featured_matches["Player1Won"].isin([0, 1]).all()

assert featured_matches["Player1WinRateBefore"].between(0, 1).all()
assert featured_matches["Player2WinRateBefore"].between(0, 1).all()

assert featured_matches["Player1SurfaceWinRateBefore"].between(0, 1).all()
assert featured_matches["Player2SurfaceWinRateBefore"].between(0, 1).all()

assert featured_matches["Player1H2HWinRateBefore"].between(0, 1).all()
assert featured_matches["Player2H2HWinRateBefore"].between(0, 1).all()

# missing-value indicators must be binary 
assert featured_matches["MissingPlayer1DaysSinceLastMatch"].isin([0, 1]).all()
assert featured_matches["MissingPlayer2DaysSinceLastMatch"].isin([0, 1]).all()

print("All validation check completed.")

All validation check completed.


In [17]:
# check for missing values 
missing_values = featured_matches[feature_columns].isna().sum()

missing_values[missing_values > 0].sort_values(ascending=False)

DaysSinceLastMatchDifference    734
Player2DaysSinceLastMatch       412
Player1DaysSinceLastMatch       392
dtype: int64

### Missing values

`Player1DaysSinceLastMatch`, `Player2DaysSinceLastMatch`, and their difference can be missing when
a player has no previous match in the available dataset. This is expected rather than an error.

The corresponding binary missing indicators explicitly record whether the previous-match date was
unavailable. The numerical missing values can then be handled during preprocessing. Any imputation
used for model training should be fitted using the training data only and then applied to later
data, so that no information from the validation or test sets is introduced.


In [ ]:
# on the first available date, no previous match should contribute to the historical features.
first_date = featured_matches["Date"].min()

featured_matches.loc[
    featured_matches["Date"].eq(first_date), 
    [
        "Date",
        "Player1",
        "Player2",
        "Player1MatchesBefore", # should be zero
        "Player2MatchesBefore", # should be zero
        "H2HMatchesBefore" # should be zero
    ]
].head(20)

In [ ]:
# select a frequently occuring player to verify how the historical feeatures evolve over time

all_players = pd.concat(
    [
        featured_matches["Player1"],
        featured_matches["Player2"]
    ],
    ignore_index=True
) # takes all the players that have played 

# we count the number of occurences for each player and choose the one that appears more frequently
example_player = all_players.value_counts().index[0]

print("Example player:", example_player)

In [ ]:
# inspect the selected player's historical features across their matches.
example_player_matches = featured_matches.loc[
    featured_matches["Player1"].eq(example_player) | featured_matches["Player2"].eq(example_player),
    [
        "Date",
        "Player1",
        "Player2",
        "Player1MatchesBefore",
        "Player2MatchesBefore",
        "Player1WinRateBefore",
        "Player2WinRateBefore",
        "Player1Won" 
    ]
].sort_values("Date")

example_player_matches.head(15)

In [ ]:
# saved the engineered dataset for use in the modelling pipeline
output_file = PROCESSED_DATA_DIR / "featured_matches_2015_2025.csv"

featured_matches.to_csv(
    output_file, 
    index = False
)

print("Featured dataset saved to:")
print(output_file.resolve())

In [ ]:
# read the saved dataset back and verify that it can be loaded successfully

saved_featured_matches = pd.read_csv(
    output_file, 
    parse_dates=["Date"]
)

print("Saved featured shape:", saved_featured_matches.shape)

saved_featured_matches[
    [
        "Date",
        "Player1",
        "Player2",
        "RankDifference",
        "WinRateDifference",
        "SurfaceWinRateDifference",
        "Player1Won" 
    ]
].head()

In [ ]:
# final summary of the engineered dataset

print("Feature shape:", featured_matches.shape)

print(featured_matches[feature_columns].isna().sum())

print(featured_matches[
        [
            "Player1Won",
            "RankDifference",
            "WinRateDifference",
            "Recent5WinRateDifference",
            "SurfaceMatchesDifference",
            "SurfaceWinRateDifference",
            "H2HMatchesBefore",
            "DaysSinceLastMatchDifference"
        ]
    ].describe()
)